# Initial Data Exploration (Chunk-Based) — 3 Months UP (0-6 Years) with Demographics

This notebook performs EDA on `3_months_UP_0m_6y_data_with_demographic_info.csv` using **chunked loading**.

The analysis is done from scratch with streaming computations so we do not rely on prior outputs or full in-memory assumptions.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter, defaultdict
from IPython.display import display

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Configuration and Schema Discovery

Set chunk size, inspect only header first, and discover columns before scanning data rows.

In [7]:
CSV_PATH = "/home/harsh_wadhwaniai_org/eda-project/data/3_months_UP_0m_6y_data_with_demographic_info.csv"
CHUNK_SIZE = 20000
REF_DATE = pd.Timestamp("2024-04-01")
DISTINCT_TRACK_LIMIT = 100000

schema_only = pd.read_csv(CSV_PATH, nrows=0)
ALL_COLUMNS = schema_only.columns.tolist()

print(f"Discovered {len(ALL_COLUMNS)} columns")
print(ALL_COLUMNS)

Discovered 29 columns
['beneficiary_id', 'dob', 'gender', 'birth_height', 'birth_weight', 'feb24_status', 'feb24_height', 'feb24_weight', 'feb24_height_weight_entered_date', 'feb24_hb', 'feb24_hb_test_date', 'mar24_status', 'mar24_height', 'mar24_weight', 'mar24_height_weight_entered_date', 'mar24_hb', 'mar24_hb_test_date', 'apr24_status', 'apr24_height', 'apr24_weight', 'apr24_height_weight_entered_date', 'apr24_hb', 'apr24_hb_test_date', 'state_id', 'district_id', 'project_id', 'sector_id', 'awc_id', 'awc_code']


## 2. Chunk Profiling Utilities

In [8]:
def detect_numeric_like_columns(sample_chunk, min_ratio=0.95):
    """Classify columns as numeric-like based on parseable ratio in first chunk."""
    numeric_like = []
    categorical_like = []

    for col in sample_chunk.columns:
        s = sample_chunk[col]
        if pd.api.types.is_numeric_dtype(s):
            numeric_like.append(col)
            continue

        non_null = s.dropna()
        if non_null.empty:
            categorical_like.append(col)
            continue

        parsed = pd.to_numeric(non_null, errors="coerce")
        ratio = parsed.notna().mean()
        if ratio >= min_ratio:
            numeric_like.append(col)
        else:
            categorical_like.append(col)

    return numeric_like, categorical_like


def init_numeric_stats(columns):
    return {
        col: {
            "count": 0,
            "sum": 0.0,
            "min": np.nan,
            "max": np.nan,
            "zero_count": 0,
        }
        for col in columns
    }


def update_numeric_stats(stats, col, series_numeric):
    vals = series_numeric.dropna()
    if vals.empty:
        return

    st = stats[col]
    st["count"] += int(vals.shape[0])
    st["sum"] += float(vals.sum())
    st["zero_count"] += int((vals == 0).sum())

    cmin = float(vals.min())
    cmax = float(vals.max())

    if np.isnan(st["min"]):
        st["min"] = cmin
    else:
        st["min"] = min(st["min"], cmin)

    if np.isnan(st["max"]):
        st["max"] = cmax
    else:
        st["max"] = max(st["max"], cmax)

In [9]:
# First chunk is used only for data-type profiling.
first_chunk = next(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=False))
NUMERIC_LIKE_COLS, CATEGORICAL_LIKE_COLS = detect_numeric_like_columns(first_chunk)

print(f"First chunk shape: {first_chunk.shape}")
print(f"Numeric-like columns detected: {len(NUMERIC_LIKE_COLS)}")
print(f"Categorical-like columns detected: {len(CATEGORICAL_LIKE_COLS)}")

schema_profile = pd.DataFrame({
    "column": ALL_COLUMNS,
    "profiled_type": [
        "numeric_like" if c in NUMERIC_LIKE_COLS else "categorical_like"
        for c in ALL_COLUMNS
    ],
})

display(schema_profile)

First chunk shape: (20000, 29)
Numeric-like columns detected: 21
Categorical-like columns detected: 8


,column,profiled_type
0,beneficiary_id,numeric_like
1,dob,categorical_like
2,gender,categorical_like
3,birth_height,numeric_like
4,birth_weight,numeric_like
5,feb24_status,categorical_like
6,feb24_height,numeric_like
7,feb24_weight,numeric_like
8,feb24_height_weight_entered_date,categorical_like
9,feb24_hb,numeric_like


## 3. Pass 1 (Chunked): Full-File Streaming Profile

In [ ]:
total_rows = 0
chunk_count = 0

missing_counts = defaultdict(int)
non_null_counts = defaultdict(int)

numeric_stats = init_numeric_stats(NUMERIC_LIKE_COLS)
value_counters = {c: Counter() for c in CATEGORICAL_LIKE_COLS}

distinct_seen = {c: set() for c in CATEGORICAL_LIKE_COLS}
distinct_overflow = {c: False for c in CATEGORICAL_LIKE_COLS}

month_prefixes = sorted({c[:-7] for c in ALL_COLUMNS if c.endswith("_status")})
month_coverage = {
    m: {
        "rows": 0,
        "status_present": 0,
        "height_gt_0": 0,
        "weight_gt_0": 0,
        "height_and_weight_gt_0": 0,
    }
    for m in month_prefixes
}

dob_invalid_count = 0
dob_non_null_count = 0
age_bins = np.arange(0, 85, 1)
age_hist = np.zeros(len(age_bins) - 1, dtype=np.int64)
age_count = 0
age_sum = 0.0
age_min = np.nan
age_max = np.nan

for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    chunk_count += 1
    n = len(chunk)
    total_rows += n

    # Missingness/non-null counts by column.
    for col in ALL_COLUMNS:
        s = chunk[col]
        missing_counts[col] += int(s.isna().sum())
        non_null_counts[col] += int(s.notna().sum())

    # Numeric profiling without loading whole file.
    for col in NUMERIC_LIKE_COLS:
        numeric_series = pd.to_numeric(chunk[col], errors="coerce")
        update_numeric_stats(numeric_stats, col, numeric_series)

    # Categorical distributions and distinct tracking with cap.
    for col in CATEGORICAL_LIKE_COLS:
        vals = chunk[col].astype("string").fillna("<MISSING>")
        value_counters[col].update(vals.value_counts(dropna=False).to_dict())

        if not distinct_overflow[col]:
            distinct_seen[col].update(vals.unique().tolist())
            if len(distinct_seen[col]) > DISTINCT_TRACK_LIMIT:
                distinct_seen[col].clear()
                distinct_overflow[col] = True

    # Month-wise coverage from discovered month prefixes.
    for m in month_prefixes:
        status_col = f"{m}_status"
        h_col = f"{m}_height"
        w_col = f"{m}_weight"

        if status_col not in chunk.columns or h_col not in chunk.columns or w_col not in chunk.columns:
            continue

        h = pd.to_numeric(chunk[h_col], errors="coerce")
        w = pd.to_numeric(chunk[w_col], errors="coerce")
        status = chunk[status_col]

        month_coverage[m]["rows"] += n
        month_coverage[m]["status_present"] += int(status.notna().sum())
        month_coverage[m]["height_gt_0"] += int((h > 0).sum(skipna=True))
        month_coverage[m]["weight_gt_0"] += int((w > 0).sum(skipna=True))
        month_coverage[m]["height_and_weight_gt_0"] += int(((h > 0) & (w > 0)).sum(skipna=True))

    # DOB parse quality + age histogram.
    if "dob" in chunk.columns:
        dob_raw = chunk["dob"]
        dob_parsed = pd.to_datetime(dob_raw, errors="coerce")

        dob_non_null_mask = dob_raw.notna()
        dob_non_null_count += int(dob_non_null_mask.sum())
        dob_invalid_count += int((dob_non_null_mask & dob_parsed.isna()).sum())

        valid_dob = dob_parsed.dropna()
        if not valid_dob.empty:
            age_months = ((REF_DATE - valid_dob).dt.days / 30.44).clip(lower=0, upper=84)
            age_hist += np.histogram(age_months, bins=age_bins)[0]

            age_count += int(age_months.shape[0])
            age_sum += float(age_months.sum())
            cmin = float(age_months.min())
            cmax = float(age_months.max())
            if np.isnan(age_min):
                age_min = cmin
                age_max = cmax
            else:
                age_min = min(age_min, cmin)
                age_max = max(age_max, cmax)

    if chunk_count % 10 == 0:
        print(f"Processed chunks: {chunk_count:,} | rows so far: {total_rows:,}")

print("\nStreaming pass complete")
print(f"Total chunks: {chunk_count:,}")
print(f"Total rows: {total_rows:,}")

## 4. Aggregated Results (From Chunked Pass)

In [ ]:
missing_df = pd.DataFrame({
    "column": ALL_COLUMNS,
    "missing_count": [missing_counts[c] for c in ALL_COLUMNS],
    "missing_pct": [round((missing_counts[c] / total_rows) * 100, 4) for c in ALL_COLUMNS],
    "non_null_count": [non_null_counts[c] for c in ALL_COLUMNS],
}).sort_values("missing_pct", ascending=False)

display(missing_df)

numeric_rows = []
for col in NUMERIC_LIKE_COLS:
    st = numeric_stats[col]
    if st["count"] == 0:
        continue
    mean_val = st["sum"] / st["count"]
    zero_pct = (st["zero_count"] / st["count"]) * 100
    numeric_rows.append({
        "column": col,
        "numeric_count": st["count"],
        "mean": mean_val,
        "min": st["min"],
        "max": st["max"],
        "zero_count": st["zero_count"],
        "zero_pct": zero_pct,
    })

numeric_summary_df = pd.DataFrame(numeric_rows).sort_values("zero_pct", ascending=False)
display(numeric_summary_df)

categorical_cardinality_rows = []
for col in CATEGORICAL_LIKE_COLS:
    tracked = not distinct_overflow[col]
    distinct_value = len(distinct_seen[col]) if tracked else f"> {DISTINCT_TRACK_LIMIT:,}"
    top_value, top_count = value_counters[col].most_common(1)[0] if value_counters[col] else (None, 0)
    categorical_cardinality_rows.append({
        "column": col,
        "distinct_values": distinct_value,
        "exact_distinct_tracked": tracked,
        "top_value": top_value,
        "top_count": top_count,
        "top_pct": round((top_count / total_rows) * 100, 4),
    })

categorical_summary_df = pd.DataFrame(categorical_cardinality_rows).sort_values("column")
display(categorical_summary_df)

## 5. Key Categorical Distributions (Top Values)

In [ ]:
important_cols = [
    "gender",
    "state_id",
    "district_id",
    "project_id",
    "sector_id",
    "awc_id",
    "awc_code",
]

important_cols += [c for c in ALL_COLUMNS if c.endswith("_status")]
important_cols = [c for c in important_cols if c in value_counters]

for col in important_cols:
    print(f"\n--- {col} (top 10) ---")
    top10 = value_counters[col].most_common(10)
    top_df = pd.DataFrame(top10, columns=["value", "count"])
    top_df["pct"] = (top_df["count"] / total_rows * 100).round(4)
    display(top_df)

In [ ]:
## 6. Month-Wise Coverage (Derived from Chunked Scan)

In [ ]:
coverage_rows = []
for m in month_prefixes:
    row_base = month_coverage[m]["rows"] if month_coverage[m]["rows"] else np.nan
    coverage_rows.append({
        "month": m,
        "rows_seen": month_coverage[m]["rows"],
        "status_present_pct": round(month_coverage[m]["status_present"] / row_base * 100, 4),
        "height_gt_0_pct": round(month_coverage[m]["height_gt_0"] / row_base * 100, 4),
        "weight_gt_0_pct": round(month_coverage[m]["weight_gt_0"] / row_base * 100, 4),
        "height_and_weight_gt_0_pct": round(month_coverage[m]["height_and_weight_gt_0"] / row_base * 100, 4),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values("month")
display(coverage_df)

plt.figure(figsize=(8, 4))
plot_df = coverage_df.set_index("month")[[
    "status_present_pct",
    "height_gt_0_pct",
    "weight_gt_0_pct",
    "height_and_weight_gt_0_pct",
]]
plot_df.plot(kind="bar", ax=plt.gca())
plt.title("Month-wise coverage percentages")
plt.ylabel("Percent")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. DOB Quality and Age Profile (From Chunked Scan)

DOB parsing is validated in-stream, and age distribution is computed from chunked histogram aggregation.

In [ ]:
invalid_dob_pct = (dob_invalid_count / dob_non_null_count * 100) if dob_non_null_count else np.nan
age_mean = (age_sum / age_count) if age_count else np.nan

print(f"DOB non-null rows: {dob_non_null_count:,}")
print(f"DOB invalid rows: {dob_invalid_count:,}")
print(f"DOB invalid pct among non-null: {invalid_dob_pct:.4f}%")

age_summary = pd.DataFrame([
    {"metric": "age_count", "value": age_count},
    {"metric": "age_mean_months", "value": age_mean},
    {"metric": "age_min_months", "value": age_min},
    {"metric": "age_max_months", "value": age_max},
])
display(age_summary)

age_bin_labels = age_bins[:-1]
plt.figure(figsize=(11, 4))
plt.bar(age_bin_labels, age_hist, width=0.9, color="steelblue")
plt.title("Approximate age distribution (months) from chunked aggregation")
plt.xlabel("Age in months")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 8. Notes

- This notebook is intentionally chunk-driven and computes metrics from streamed data.
- Distinct counts for very high-cardinality categorical columns are tracked exactly only until `DISTINCT_TRACK_LIMIT`; beyond that, they are marked as overflow.
- Any interpretation should use the displayed computed tables (missingness, numeric profile, categorical distributions, coverage, DOB quality) from the current run.

## 9. Create Cleaned CSV (Drop Hb Columns + Filter Null `dob`/`gender`)

This section processes the original CSV in chunks, drops all columns containing `hb` in the name, removes rows where `dob` or `gender` is null/blank, and writes a cleaned CSV file.

In [ ]:
from pathlib import Path
import csv

SOURCE_CSV = Path("/home/harsh_wadhwaniai_org/eda-project/data/3_months_UP_0m_6y_data_with_demographic_info.csv")
CLEANED_CSV = Path("/home/harsh_wadhwaniai_org/eda-project/data/3_months_UP_0m_6y_data_with_demographic_info_no_hb_dob_gender_clean.csv")
CHUNK_SIZE_CLEAN = 200_000

print("Source:", SOURCE_CSV)
print("Cleaned output:", CLEANED_CSV)
print("Chunk size:", CHUNK_SIZE_CLEAN)

In [ ]:
# Discover columns first and identify hb columns to drop.
header_df = pd.read_csv(SOURCE_CSV, nrows=0)
source_columns = header_df.columns.tolist()

hb_columns = [c for c in source_columns if "hb" in c.lower()]
keep_columns = [c for c in source_columns if c not in hb_columns]

print(f"Original columns: {len(source_columns)}")
print(f"Dropping hb columns: {len(hb_columns)}")
print(hb_columns)
print(f"Output columns: {len(keep_columns)}")

In [ ]:
rows_total = 0
rows_kept = 0
rows_dropped_null_birth_height = 0
rows_dropped_null_birth_weight = 0
rows_dropped_null_birth_height_or_weight = 0

CLEANED_CSV.parent.mkdir(parents=True, exist_ok=True)

first_chunk = True
for chunk in pd.read_csv(SOURCE_CSV, chunksize=CHUNK_SIZE_CLEAN, low_memory=False):
    rows_total += len(chunk)

    birth_height_missing     = chunk["birth_height"].isna() | (chunk["birth_height"].astype("string").str.strip() == "")
    birth_weight_missing = chunk["birth_weight"].isna() | (chunk["birth_weight"].astype("string").str.strip() == "")
    drop_mask = birth_height_missing | birth_weight_missing

    rows_dropped_null_birth_height += int(birth_height_missing.sum())
    rows_dropped_null_birth_weight += int(birth_weight_missing.sum())
    rows_dropped_null_birth_height_or_weight += int(drop_mask.sum())

    cleaned_chunk = chunk.loc[~drop_mask, keep_columns].copy()
    rows_kept += len(cleaned_chunk)

    cleaned_chunk.to_csv(
        CLEANED_CSV,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )
    first_chunk = False

print("Chunked cleaning complete.")

In [ ]:
retention_pct = (rows_kept / rows_total * 100) if rows_total else 0.0

summary_df = pd.DataFrame([
    {"metric": "rows_total", "value": rows_total},
    {"metric": "rows_dropped_null_birth_height", "value": rows_dropped_null_birth_height},
    {"metric": "rows_dropped_null_birth_weight", "value": rows_dropped_null_birth_weight},
    {"metric": "rows_dropped_null_birth_height_or_weight", "value": rows_dropped_null_birth_height_or_weight},
    {"metric": "rows_remaining", "value": rows_kept},
    {"metric": "retention_pct", "value": round(retention_pct, 4)},
    {"metric": "original_column_count", "value": len(source_columns)},
    {"metric": "dropped_hb_column_count", "value": len(hb_columns)},
    {"metric": "output_column_count", "value": len(keep_columns)},
])

display(summary_df)
print("Saved cleaned CSV:", CLEANED_CSV)

In [ ]:
# Optional quick validation read.
validation_sample = pd.read_csv(CLEANED_CSV, nrows=5)
print("Validation sample shape:", validation_sample.shape)
display(validation_sample.head())

In [ ]:
data = pd.read_csv(CLEANED_CSV)

In [ ]:
len(data)

## Add Nutrition Labels To Cleaned Dataset

Use `src/nutrition_labels.py` lookup logic to create month-wise stunting, underweight, and wasting labels for every row in the cleaned CSV.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

CLEANED_CSV='/home/harsh_wadhwaniai_org/eda-project/data/cleaned_dataset.csv'

PROJECT_ROOT = Path("/home/harsh_wadhwaniai_org/eda-project")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.nutrition_labels import classify_all

# Prefer the notebook's cleaned output if present; otherwise use cleaned_dataset.csv
if "CLEANED_CSV" in globals() and Path(CLEANED_CSV).exists():
    INPUT_CSV = Path(CLEANED_CSV)
else:
    INPUT_CSV = PROJECT_ROOT / "data" / "cleaned_dataset.csv"

OUTPUT_CSV = PROJECT_ROOT / "data" / "cleaned_dataset_with_labels.csv"
CHUNK_SIZE = 100_000

# Fallback dates if entered_date is missing
MONTH_CONFIG = {
    "feb24": pd.Timestamp("2024-02-01"),
    "mar24": pd.Timestamp("2024-03-01"),
    "apr24": pd.Timestamp("2024-04-01"),
}


def _safe_float(value):
    if pd.isna(value):
        return None
    try:
        value = float(value)
        return value if value > 0 else None
    except Exception:
        return None


def _normalize_sex(value):
    if pd.isna(value):
        return "M"
    v = str(value).strip().upper()
    if v in {"M", "MALE", "BOY"}:
        return "M"
    if v in {"F", "FEMALE", "GIRL"}:
        return "F"
    return "M"


def _compute_age_days(dob_value, entered_date_value, fallback_date):
    dob = pd.to_datetime(dob_value, errors="coerce")
    if pd.isna(dob):
        return None

    measured_on = pd.to_datetime(entered_date_value, errors="coerce")
    if pd.isna(measured_on):
        measured_on = fallback_date

    age_days = (measured_on - dob).days
    if pd.isna(age_days):
        return None
    return max(int(age_days), 0)


def _label_single_measurement(gender, dob, entered_date, month_key, height_value, weight_value):
    # Keep local name explicit to avoid confusion with any stale outer variables.
    age_days_value = _compute_age_days(dob, entered_date, MONTH_CONFIG[month_key])
    if age_days_value is None:
        return {
            "stunting_status": None,
            "is_stunted": False,
            "underweight_status": None,
            "is_underweight": False,
            "wasting_status": None,
            "is_wasted": False,
            "is_sam": False,
        }

    sex = _normalize_sex(gender)
    height_cm = _safe_float(height_value)
    weight_kg = _safe_float(weight_value)

    return classify_all(
        sex=sex,
        age_days=int(age_days_value),
        height_cm=height_cm,
        weight_kg=weight_kg,
    )


OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
if OUTPUT_CSV.exists():
    OUTPUT_CSV.unlink()

df = pd.read_csv(INPUT_CSV, low_memory=False)

In [ ]:
! pip install tqdm

In [ ]:
! pip install ipywidg

In [12]:
try:
    from tqdm.notebook import tqdm as tqdm_bar
except Exception:
    from tqdm import tqdm as tqdm_bar

print(f"Loaded rows: {len(df):,}")

for month in MONTH_CONFIG:
    h_col = f"{month}_height"
    w_col = f"{month}_weight"
    d_col = f"{month}_height_weight_entered_date"

    labels = []
    total = len(df)

    iterator = zip(df["gender"], df["dob"], df[d_col], df[h_col], df[w_col])
    for g, dob, d, h, w in tqdm_bar(
        iterator,
        total=total,
        desc=f"Labeling {month}",
        unit="rows",
        leave=True,
    ):
        labels.append(_label_single_measurement(g, dob, d, month, h, w))

    df[f"{month}_stunting_status"] = [x["stunting_status"] for x in labels]
    df[f"{month}_is_stunted"] = [x["is_stunted"] for x in labels]
    df[f"{month}_underweight_status"] = [x["underweight_status"] for x in labels]
    df[f"{month}_is_underweight"] = [x["is_underweight"] for x in labels]
    df[f"{month}_wasting_status"] = [x["wasting_status"] for x in labels]
    df[f"{month}_is_wasted"] = [x["is_wasted"] for x in labels]
    df[f"{month}_is_sam"] = [x["is_sam"] for x in labels]

    print(f"Completed month: {month}")

df.to_csv(OUTPUT_CSV, index=False)
print(f"\nDone. Saved {len(df):,} rows to: {OUTPUT_CSV}")

Loaded rows: 3,637,040


Labeling feb24:   0%|          | 0/3637040 [00:00<?, ?rows/s]

KeyboardInterrupt: 

# Clean Dataset with Labels

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/home/harsh_wadhwaniai_org/eda-project/data/cleaned_dataset_with_labels.csv')

/tmp/ipykernel_7471/2419000737.py:1: DtypeWarning: Columns (0: apr24_status, 1: apr24_height_weight_entered_date, 2: feb24_wasting_status, 3: mar24_wasting_status, 4: apr24_stunting_status, 5: apr24_underweight_status, 6: apr24_wasting_status) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/harsh_wadhwaniai_org/eda-project/data/cleaned_dataset_with_labels.csv')


In [9]:
df.head()

,beneficiary_id,dob,gender,birth_height,birth_weight,feb24_status,feb24_height,feb24_weight,feb24_height_weight_entered_date,mar24_status,...,mar24_wasting_status,mar24_is_wasted,mar24_is_sam,apr24_stunting_status,apr24_is_stunted,apr24_underweight_status,apr24_is_underweight,apr24_wasting_status,apr24_is_wasted,apr24_is_sam
0,276437090,2023-06-25,M,31.0,2.5,active,0.0,0.0,NaN,active,...,NaN,False,False,NaN,False,NaN,False,NaN,False,False
1,282289533,2023-08-07,F,55.0,2.0,active,0.0,0.0,NaN,active,...,NaN,False,False,NaN,False,NaN,False,NaN,False,False
2,284550121,2023-10-02,M,59.0,3.5,active,0.0,0.0,NaN,active,...,NaN,False,False,NaN,False,NaN,False,NaN,False,False
3,284153753,2023-09-12,F,56.0,2.9,active,0.0,0.0,NaN,active,...,NaN,False,False,NaN,False,NaN,False,NaN,False,False
4,286058723,2023-09-01,M,36.2,2.8,active,0.0,0.0,NaN,active,...,NaN,False,False,NaN,False,NaN,False,NaN,False,False


In [6]:
! pip install seaborn

In [10]:
df.columns

Index(['beneficiary_id', 'dob', 'gender', 'birth_height', 'birth_weight',
       'feb24_status', 'feb24_height', 'feb24_weight',
       'feb24_height_weight_entered_date', 'mar24_status', 'mar24_height',
       'mar24_weight', 'mar24_height_weight_entered_date', 'apr24_status',
       'apr24_height', 'apr24_weight', 'apr24_height_weight_entered_date',
       'state_id', 'district_id', 'project_id', 'sector_id', 'awc_id',
       'awc_code', 'feb24_stunting_status', 'feb24_is_stunted',
       'feb24_underweight_status', 'feb24_is_underweight',
       'feb24_wasting_status', 'feb24_is_wasted', 'feb24_is_sam',
       'mar24_stunting_status', 'mar24_is_stunted', 'mar24_underweight_status',
       'mar24_is_underweight', 'mar24_wasting_status', 'mar24_is_wasted',
       'mar24_is_sam', 'apr24_stunting_status', 'apr24_is_stunted',
       'apr24_underweight_status', 'apr24_is_underweight',
       'apr24_wasting_status', 'apr24_is_wasted', 'apr24_is_sam'],
      dtype='str')